# HMM Basics: Construction, Generation, and I/O

This notebook covers the fundamentals of working with hidden Markov models in
conin: creating a model from probability dictionaries, generating sequences,
scoring them, and saving/loading models to disk.

See also the [constraints](constraints.ipynb) and
[inference](inference.ipynb) notebooks, which cover all model types
including HMMs.

In [ ]:
import os
import tempfile

from conin.hidden_markov_model import HiddenMarkovModel, random_hmm

## Creating an HMM

An HMM in conin is specified by three probability dictionaries:

| Parameter | Keys | Values |
| --- | --- | --- |
| `start_probs` | hidden-state labels | P(initial state) |
| `transition_probs` | `(from_state, to_state)` tuples | P(transition) |
| `emission_probs` | `(hidden_state, observed_state)` tuples | P(emission) |

State labels can be any hashable Python object — strings are the most common
choice.

In [ ]:
start_probs = {"sunny": 0.6, "rainy": 0.4}

transition_probs = {
    ("sunny", "sunny"): 0.7,
    ("sunny", "rainy"): 0.3,
    ("rainy", "sunny"): 0.4,
    ("rainy", "rainy"): 0.6,
}

emission_probs = {
    ("sunny", "walk"): 0.6,
    ("sunny", "shop"): 0.3,
    ("sunny", "clean"): 0.1,
    ("rainy", "walk"): 0.1,
    ("rainy", "shop"): 0.4,
    ("rainy", "clean"): 0.5,
}

hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs=start_probs,
    transition_probs=transition_probs,
    emission_probs=emission_probs,
)

Once loaded, the model exposes the state labels and counts:

In [ ]:
print("Hidden states:  ", hmm.hidden_states)
print("Observed states:", hmm.observed_states)
print("# hidden:       ", hmm.num_hidden_states)
print("# observed:     ", hmm.num_observed_states)

The probability dictionaries can be retrieved in the same format used by
`load_model`:

In [ ]:
print("Start probs:     ", hmm.get_start_probs())
print("Transition probs:", hmm.get_transition_probs())
print("Emission probs:  ", hmm.get_emission_probs())

`print()` gives a compact summary of the full model:

In [ ]:
print(hmm)

## Random HMMs

`random_hmm` creates a model with randomly generated probabilities — useful for
quick experiments and testing. You specify the state labels and an optional seed
for reproducibility.

In [ ]:
rhmm = random_hmm(
    hidden_states=["A", "B", "C"],
    observed_states=["x", "y"],
    seed=42,
)

print("Hidden states:  ", rhmm.hidden_states)
print("Observed states:", rhmm.observed_states)
print()
print(rhmm)

## File I/O

Models can be saved to and loaded from JSON files with `write_to_file` and
`read_from_file`.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "weather_hmm.json")

    hmm.write_to_file(path)
    print("Saved to", path)

    hmm2 = HiddenMarkovModel()
    hmm2.read_from_file(path)

    print("Loaded hidden states:", hmm2.hidden_states)
    print("Start probs match:   ", hmm.get_start_probs() == hmm2.get_start_probs())

## Generating Data

An HMM defines a joint distribution over hidden and observed sequences.
Several generation methods let you sample from it.

### Hidden sequences

`generate_hidden` samples a hidden-state sequence of a given length:

In [ ]:
hmm.set_seed(0)

hidden = hmm.generate_hidden(10)
print("Hidden:", hidden)

### Observed sequences conditioned on hidden

`generate_observed_from_hidden` samples observations given a known hidden path:

In [ ]:
observed = hmm.generate_observed_from_hidden(hidden)
print("Hidden:  ", hidden)
print("Observed:", observed)

### Observed sequences (marginal)

`generate_observed` samples an observed sequence directly — the hidden states
are generated internally but not returned:

In [ ]:
observed_only = hmm.generate_observed(10)
print("Observed:", observed_only)

### Sampling until a target state

`generate_hidden_until_state` keeps sampling until a specific hidden state is
reached. The length of the resulting sequence is random:

In [ ]:
hmm.set_seed(4)

path = hmm.generate_hidden_until_state("rainy")
print(f"Path (length {len(path)}):", path)
print("Ends in 'rainy':", path[-1] == "rainy")

## Scoring Sequences

`log_probability` computes the joint log-probability of an aligned hidden and
observed sequence under the model. This is useful for comparing how well
different hidden paths explain the same observations.

In [ ]:
hmm.set_seed(1)

hidden = hmm.generate_hidden(8)
observed = hmm.generate_observed_from_hidden(hidden)

logp = hmm.log_probability(observed, hidden)

print("Hidden:  ", hidden)
print("Observed:", observed)
print(f"Log-probability: {logp:.4f}")

A random hidden path for the same observations will typically score worse:

In [ ]:
random_hidden = hmm.generate_hidden(len(observed))

logp_random = hmm.log_probability(observed, random_hidden)

print("Random hidden: ", random_hidden)
print(f"Log-probability: {logp_random:.4f}")
print(f"Difference:      {logp - logp_random:+.4f}  (original is better when positive)")